# Replicação do Estudo H-CAT para Dataset de Covertype

Este notebook replica os experimentos do paper "Dissecting Sample Hardness: A Fine-Grained Analysis of Hardness Characterization Methods for Data-Centric AI" para o dataset de covertype.

## Objetivo
Aplicar o H-CAT no dataset tabular de covertype seguindo a mesma metodologia do paper.

## Otimizações Aplicadas
- **Amostragem**: Dataset limitado a 20k amostras (estratificada) para acelerar execução
- **Batch size maior**: 128 (ao invés de 64) para processar mais rápido
- **Processamento em células separadas**: Cada tipo de hardness em célula própria para evitar execuções >2h
- **Épocas reduzidas**: 5 épocas (mantém qualidade, reduz tempo)

## Correções Aplicadas
1. **CustomDataset.__getitem__**: Agora usa `self.targets` diretamente e garante que target seja sempre um número Python nativo
2. **PerturbedDataset.__getitem__**: Removido assert problemático, agora usa `self.perturbs` diretamente com tratamento de erro robusto
3. **_generate_mislabels()**: Garante que as chaves do dict sejam int (não numpy.int64) para evitar problemas de tipo
4. **Atypical perturbation**: Corrigido problema de array read-only

In [ ]:
import sys
from pathlib import Path

import numpy as np
import openml
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

hcat_path = Path("../H-CAT").resolve()
if str(hcat_path) not in sys.path:
    sys.path.insert(0, str(hcat_path))

from src.trainer import PyTorchTrainer
from src.dataloader import MultiFormatDataLoader
from src.models import MLP
from src.evaluator import Evaluator
from src.utils import seed_everything


## 2. Parâmetros do Experimento

Configurações baseadas no paper e no script `run_experiment_tabular.py`, otimizadas para execução mais rápida.


In [ ]:
TOTAL_RUNS = 3
EPOCHS = 10
SEED = 0
MAX_SAMPLE_SIZE = 50000
BATCH_SIZE = 64

HARDNESS_TYPES = ["uniform", "asymmetric", "adjacent", "instance", "atypical"]
PROPORTIONS = [0.1, 0.3, 0.5]

# Rule matrix e feature usadas por 'instance' e 'atypical' (paper H-CAT, classes 0-6).
RULE_MATRIX_COVERTYPE = {4: [5], 1: [0], 0: [6], 6: [0], 2: [3], 5: [2], 3: [2]}
ATYPICAL_FEATURE = "Elevation"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 3. Carregamento e Validação do Dataset

O dataset de covertype é carregado do OpenML (ID: 1596). Aplicamos amostragem estratificada para limitar o tamanho e acelerar a execução.


In [ ]:
dataset = openml.datasets.get_dataset(1596)
X, y, _, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute,
)
df = pd.DataFrame(X, columns=attribute_names).dropna()
# Covertype tem classes 1-7; H-CAT espera 0-indexado.
df["y"] = LabelEncoder().fit_transform(y.loc[df.index])

if len(df) > MAX_SAMPLE_SIZE:
    df = df.sample(MAX_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)


## 4. Função para Executar Experimentos

Esta seção define a função `run_experiment` que executa um experimento completo do H-CAT.

In [ ]:
def run_experiment(hardness_type, proportion, run_number, seed, df, device):
    seed_everything(seed)
    X = df.drop(columns="y").to_numpy()
    y = df["y"].values
    num_classes = len(np.unique(y))

    rule_matrix = RULE_MATRIX_COVERTYPE if hardness_type == "instance" else None
    atypical_marginal = None
    if hardness_type == "atypical":
        marginal = df[ATYPICAL_FEATURE].values
        index = df.columns.get_loc(ATYPICAL_FEATURE)
        atypical_marginal = (marginal, index)

    dataloader_class = MultiFormatDataLoader(
        data=(X, y),
        target_column=None,
        data_type="numpy",
        data_modality="tabular",
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        transform=None,
        image_transform=None,
        perturbation_method=hardness_type,
        p=proportion,
        rule_matrix=rule_matrix,
        atypical_marginal=atypical_marginal,
    )
    dataloader, dataloader_unshuffled = dataloader_class.get_dataloader()
    flag_ids = dataloader_class.get_flag_ids()

    methods = [
        "aum", "data_uncert", "el2n", "grand", "cleanlab",
        "forgetting", "vog", "prototypicality", "loss", "conf_agree",
    ]
    model = MLP(input_size=X.shape[1], nlabels=num_classes).to(device)
    trainer = PyTorchTrainer(
        model=model,
        criterion=nn.CrossEntropyLoss(),
        optimizer=optim.Adam(model.parameters(), lr=1e-3),
        lr=1e-3,
        epochs=EPOCHS,
        total_samples=len(df),
        num_classes=num_classes,
        device=device,
        characterization_methods=methods,
    )
    trainer.fit(dataloader, dataloader_unshuffled)

    hardness_dict = trainer.get_hardness_methods()
    eval_dict, raw_dict = {}, {}
    for method, obj in hardness_dict.items():
        try:
            ev = Evaluator(
                hardness_dict={method: obj}, flag_ids=flag_ids, p=proportion,
            )
            m_eval, m_raw = ev.compute_results()
            eval_dict.update(m_eval)
            raw_dict.update(m_raw)
        except Exception:
            continue

    return {
        "hardness_type": hardness_type,
        "proportion": proportion,
        "run": run_number,
        "seed": seed,
        "metrics": eval_dict,
        "raw_scores": raw_dict,
        "flag_ids": flag_ids,
    }


## 5. Execução dos Experimentos

**IMPORTANTE**: Os experimentos estão divididos em células separadas por tipo de hardness para evitar execuções muito longas (>2 horas). Execute cada célula individualmente.

### Estrutura:
- **Célula 5.1**: Uniform mislabeling
- **Célula 5.2**: Asymmetric mislabeling  
- **Célula 5.3**: Adjacent mislabeling
- **Célula 5.4**: Instance-specific mislabeling
- **Célula 5.5**: Atypical perturbation

Cada célula processa todas as proporções e runs para aquele tipo de hardness.


In [ ]:
results_uniform = []
for prop in PROPORTIONS:
    for run in range(1, TOTAL_RUNS + 1):
        seed = SEED + (run - 1)
        try:
            result = run_experiment("uniform", prop, run, seed, df, device)
        except Exception:
            import traceback
            traceback.print_exc()
            continue
        results_uniform.append(result)


In [ ]:
results_asymmetric = []
for prop in PROPORTIONS:
    for run in range(1, TOTAL_RUNS + 1):
        seed = SEED + (run - 1)
        try:
            result = run_experiment("asymmetric", prop, run, seed, df, device)
        except Exception:
            import traceback
            traceback.print_exc()
            continue
        results_asymmetric.append(result)


In [ ]:
results_adjacent = []
for prop in PROPORTIONS:
    for run in range(1, TOTAL_RUNS + 1):
        seed = SEED + (run - 1)
        try:
            result = run_experiment("adjacent", prop, run, seed, df, device)
        except Exception:
            import traceback
            traceback.print_exc()
            continue
        results_adjacent.append(result)


In [ ]:
results_instance = []
for prop in PROPORTIONS:
    for run in range(1, TOTAL_RUNS + 1):
        seed = SEED + (run - 1)
        try:
            result = run_experiment("instance", prop, run, seed, df, device)
        except Exception:
            import traceback
            traceback.print_exc()
            continue
        results_instance.append(result)


In [ ]:
results_atypical = []
for prop in PROPORTIONS:
    for run in range(1, TOTAL_RUNS + 1):
        seed = SEED + (run - 1)
        try:
            result = run_experiment("atypical", prop, run, seed, df, device)
        except Exception:
            import traceback
            traceback.print_exc()
            continue
        results_atypical.append(result)


## 6. Consolidação dos Resultados

Consolidar todos os resultados em uma única lista.


In [ ]:
all_results = (
    results_uniform + results_asymmetric + results_adjacent
    + results_instance + results_atypical
)


## 7. Análise e Visualização dos Resultados


In [ ]:
results_list = []
for result in all_results:
    for method, metrics in result["metrics"].items():
        results_list.append({
            "hardness_type": result["hardness_type"],
            "proportion": result["proportion"],
            "run": result["run"],
            "method": method,
            "auc_roc": metrics.get("auc_roc", np.nan),
            "auc_prc": metrics.get("auc_prc", np.nan),
        })
results_df = pd.DataFrame(results_list)


In [ ]:
OUTPUT_DIR = Path("..") / "data" / "comparison_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "nb05_hcat_covertype_results.csv"

results_to_save = results_df.rename(columns={"method": "measure"}).copy()
results_to_save.insert(0, "dataset", "covertype")
results_to_save.insert(1, "method_family", "hcat")
results_to_save = results_to_save[[
    "dataset", "method_family", "hardness_type", "proportion", "run", "measure",
    "auc_roc", "auc_prc",
]]
results_to_save.to_csv(OUTPUT_CSV, index=False)


In [ ]:
summary_hardness = (
    results_df
    .groupby("hardness_type")
    .agg({"auc_roc": ["mean", "std"], "auc_prc": ["mean", "std"]})
    .round(4)
)
summary_proportion = (
    results_df
    .groupby("proportion")
    .agg({"auc_roc": ["mean", "std"], "auc_prc": ["mean", "std"]})
    .round(4)
)
summary_method = (
    results_df
    .groupby("method")
    .agg({"auc_roc": ["mean", "std"], "auc_prc": ["mean", "std"]})
    .round(4)
)
summary_hardness, summary_proportion, summary_method


In [ ]:
# Visualizações
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

for hardness_type in HARDNESS_TYPES:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    hardness_data = results_df[results_df["hardness_type"] == hardness_type]
    
    if len(hardness_data) == 0:
        print(f"⚠️  Nenhum dado disponível para {hardness_type}")
        continue
    
    ax1 = axes[0]
    for method in hardness_data["method"].unique():
        method_data = hardness_data[hardness_data["method"] == method]
        mean_auc = method_data.groupby("proportion")["auc_roc"].mean()
        std_auc = method_data.groupby("proportion")["auc_roc"].std()
        ax1.errorbar(mean_auc.index, mean_auc.values, yerr=std_auc.values, 
                    label=method, marker='o', capsize=3)
    
    ax1.set_xlabel("Proporção de Perturbação")
    ax1.set_ylabel("AUC-ROC")
    ax1.set_title(f"AUC-ROC por Método - {hardness_type}")
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    for method in hardness_data["method"].unique():
        method_data = hardness_data[hardness_data["method"] == method]
        mean_auc = method_data.groupby("proportion")["auc_prc"].mean()
        std_auc = method_data.groupby("proportion")["auc_prc"].std()
        ax2.errorbar(mean_auc.index, mean_auc.values, yerr=std_auc.values, 
                    label=method, marker='s', capsize=3)
    
    ax2.set_xlabel("Proporção de Perturbação")
    ax2.set_ylabel("AUC-PRC")
    ax2.set_title(f"AUC-PRC por Método - {hardness_type}")
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Tabela Resumo Final


In [ ]:
comparison_data = []
for hardness_type in HARDNESS_TYPES:
    for prop in PROPORTIONS:
        subset = results_df[
            (results_df["hardness_type"] == hardness_type)
            & (results_df["proportion"] == prop)
        ]
        for method, auc_prc in subset.groupby("method")["auc_prc"].mean().items():
            comparison_data.append({
                "hardness_type": hardness_type,
                "proportion": prop,
                "method": method,
                "mean_auc_prc": auc_prc,
            })

comparison_df = pd.DataFrame(comparison_data)
pivot_table = comparison_df.pivot_table(
    values="mean_auc_prc",
    index=["hardness_type", "method"],
    columns="proportion",
    aggfunc="mean",
)
pivot_table.round(4)
